# 6. Feature classification  <a id="6"></a>

In this section we will train and analyse Random Forest (RF) classifier to investigate what features are most significant in predicting predominant activation loop conformational changes.

The class `FeatureClassification` facilitates running all the steps required to train a RF classifier.

In [ ]:
# Classification Analysis using FeatureClassification class
from workflow.feature_classification import FeatureClassification

# Use the filtered feature matrix and labels from feature selection (full dataset, no balancing)
classifier = FeatureClassification(
    feature_matrix=fs.feature_matrix,
    labels=fs.labels,
    unique_pairs=fs.unique_pairs,
    fully_conserved=fs.fully_conserved,
    structure_names=fs.structure_names
)

print(f"✅ FeatureClassification initialized")
print(f"   Features: {len(classifier.unique_pairs)}")
print(f"   Structures: {len(classifier.labels)}")
print(f"   Classes: {np.unique(classifier.labels)}")
print(f"   Class distribution: {dict(zip(*np.unique(classifier.labels, return_counts=True)))}")


Let's first split the data into training and validation.

In [ ]:
# Step 1: Split data into train/test sets
classifier.split_data(train_size=0.9, random_state=42)

We can now train the model with our input features.

In [ ]:
# Step 2: Train Random Forest model
classifier.train_model(n_estimators=100, random_state=42)

Let's evaluate model performance and visualise it with a confusion matrix to make sure our classifier is able to deal with the input.

In [ ]:
# Step 3: Evaluate model performance
metrics = classifier.evaluate_model()

# Step 4: Plot confusion matrix
cm = classifier.plot_confusion_matrix()

Let's now visualise and investigate what are the most significant features both looking at Mean Decrease in Impurity (MDI) and SHAP values.

In [ ]:
# Step 5: Compute feature importances (MDI)
importances, importances_std, importances_sem = classifier.compute_feature_importances()

# Step 6: Print top features
top_indices = classifier.print_top_features(n_top=20)

# Step 7: Plot feature ranking
classifier.plot_feature_ranking(n_top=20)

# Step 8: Compute permutation importances
perm_result = classifier.compute_permutation_importances(n_repeats=10, n_jobs=4)

# Step 9: Compute SHAP values (can be slow)
shap_values = classifier.compute_shap_values()
classifier.plot_shap_summary(class_idx=0, max_display=20)  # Class 0
classifier.plot_shap_summary(class_idx=1, max_display=20)  # Class 1
classifier.plot_feature_distributions(n_top=20, class_idx=1)

print("\n" + "="*60)
print("✅ CLASSIFICATION ANALYSIS COMPLETE")
print("="*60)

# 7. Top-feature distributions and W/KL separation  <a id="7"></a>

Quantify how well filtered distance features separate structures by:

1. **PCA hierarchical cluster** (cluster 0 vs 1)
2. **KinCore activation state** (inactive vs active)

For each comparison we:

- plot **split histogram violins** (train left / validation right) for the top features by **RF MDI** and **permutation** importance
- compute **Wasserstein** distance and symmetric histogram **KL**, ordered by the same RF rankings

Uses the **random** `train_test_split` from §6 (`random_state=42`, `train_size=0.9`) — not a Newick guide-tree split.

Runs on **side-chain** distances first, then the **Cα** matrix.

## 7.1 Setup: paths, KinCore labels, side-chain matrix

Load PCA cluster labels and KinCore active/inactive labels aligned to the structures used by the trained RF classifier.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from workflow.feature_classification import FeatureClassification
from workflow.pca_analysis import ClusterAnalyzer

# ── Configuration ─────────────────────────────────────────────────────────
N_TOP_VIOLINS = 20
N_BINS_KL = 31
N_WKL_FEATURES = None  # None => all features, ordered by RF importance

PCA_CLUSTER_LABELS = "cluster_labels_my_analysis_hierarchical.txt"
if not os.path.isfile(PCA_CLUSTER_LABELS):
    PCA_CLUSTER_LABELS = "Results/activation_segments/cluster_labels_my_analysis_hierarchical.txt"

KINCORE_CSV = "Results/dunbrack_assignments/kinase_conformation_assignments.csv"
WKL_ROOT = Path("Results/wkl_analysis")
WKL_SC_DIR = WKL_ROOT / "sidechain"
WKL_CA_DIR = WKL_ROOT / "ca"
WKL_SC_DIR.mkdir(parents=True, exist_ok=True)
WKL_CA_DIR.mkdir(parents=True, exist_ok=True)

# Side-chain matrix from the trained classifier (preferred) or filtered CSV
structure_names = list(classifier.structure_names) if classifier.structure_names else None
if structure_names is None or len(structure_names) != len(classifier.labels):
    if os.path.isfile("filtered_feature_matrix.csv"):
        _tmp = pd.read_csv("filtered_feature_matrix.csv", index_col=0)
        structure_names = list(_tmp.index)
    elif os.path.isfile("corr_filtered_feature_matrix.csv"):
        _tmp = pd.read_csv("corr_filtered_feature_matrix.csv", index_col=0)
        structure_names = list(_tmp.index)
    else:
        structure_names = [f"struct_{i}" for i in range(len(classifier.labels))]

X_df_sc = pd.DataFrame(
    classifier.feature_matrix,
    index=structure_names,
    columns=classifier.feature_names,
)
Xk_sc = X_df_sc.values.astype(float)
feature_labels_sc = [str(c) for c in X_df_sc.columns]
y_sc = np.asarray(classifier.labels)

if classifier.feature_importances is None:
    raise RuntimeError("Run compute_feature_importances() before §7")
if classifier.permutation_result is None:
    raise RuntimeError("Run compute_permutation_importances() before §7")
if classifier.split_labels is None:
    raise RuntimeError("Re-run split_data() so train_idx / split_labels are stored")

gini_mean_sc = np.asarray(classifier.feature_importances, dtype=float)
perm_mean_sc = np.asarray(classifier.permutation_result.importances_mean, dtype=float)
split_labels_sc = np.asarray(classifier.split_labels)
train_idx_sc = np.asarray(classifier.train_idx)
test_idx_sc = np.asarray(classifier.test_idx)
best_k_sc = Xk_sc.shape[1]

# KinCore active/inactive → CSV consumed by distribution helpers
cluster_analyzer = ClusterAnalyzer(n_clusters=2)
bio_labels_sc, _ = cluster_analyzer.load_kincore_labels(
    list(X_df_sc.index), kincore_file=KINCORE_CSV
)
bio_labels_sc = np.asarray(bio_labels_sc, dtype=float)
KINCORE_BIO_CSV = str(WKL_ROOT / "kincore_bio_labels.csv")
pd.DataFrame({"structure": X_df_sc.index, "label": bio_labels_sc.astype(int)}).to_csv(
    KINCORE_BIO_CSV, index=False
)

print(f"Side-chain matrix: {X_df_sc.shape[0]} structures × {X_df_sc.shape[1]} features")
print(f"PCA labels file: {PCA_CLUSTER_LABELS}")
print(f"KinCore CSV: {KINCORE_CSV}")
print(f"KinCore active/inactive: {int((bio_labels_sc == 1).sum())} active, "
      f"{int((bio_labels_sc == 0).sum())} inactive")
print(f"Train/val sizes: {len(train_idx_sc)} / {len(test_idx_sc)}")
print(f"W/KL output: {WKL_SC_DIR}")

## 7.2 Side-chain: split violins by PCA cluster

Top features by MDI and permutation importance; x-axis = PCA cluster.

In [ ]:
dist_sc_cluster = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
    X_df=X_df_sc,
    Xk=Xk_sc,
    feature_labels_all=feature_labels_sc,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    biological_labels_csv=KINCORE_BIO_CSV,
    pca_cluster_labels_file=PCA_CLUSTER_LABELS,
    split_labels=split_labels_sc,
    train_idx=train_idx_sc,
    test_idx=test_idx_sc,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(side-chain RF; k={best_k_sc})",
)
print("✅ Side-chain PCA-cluster distribution plots complete")

## 7.3 Side-chain: split violins by KinCore activation

Same top RF features; x-axis = inactive vs active.

In [ ]:
dist_sc_activation = FeatureClassification.plot_top_feature_distributions_by_activation(
    X_df=X_df_sc,
    Xk=Xk_sc,
    feature_labels_all=feature_labels_sc,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    biological_labels_csv=KINCORE_BIO_CSV,
    split_labels=split_labels_sc,
    train_idx=train_idx_sc,
    test_idx=test_idx_sc,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(side-chain RF; k={best_k_sc})",
)
# Keep cluster_binary from PCA distribution results for W/KL
dist_sc_activation["cluster_binary"] = dist_sc_cluster["cluster_binary"]
print("✅ Side-chain KinCore activation distribution plots complete")

## 7.4 Side-chain: Wasserstein & KL (RF-ranked)

Line plots of train/val separation vs RF importance rank for PCA clusters and KinCore labels (MDI and permutation orderings).

In [ ]:
def _run_wkl_plots(
    *,
    X_df,
    dist_cluster,
    dist_activation,
    gini_mean,
    perm_mean,
    best_k,
    out_dir: Path,
    tag: str,
):
    n_pool = X_df.shape[1] if N_WKL_FEATURES is None else min(N_WKL_FEATURES, X_df.shape[1])
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    jobs = [
        ("mdi", gini_mean, "mdi_rank", "MDI (Gini) rank (1 = highest)", "MDI-ranked"),
        ("perm", perm_mean, "perm_rank", "Permutation rank (1 = highest)", "permutation-ranked"),
    ]

    results = {}
    for key, importance, rank_col, rank_xlabel, pool_caption in jobs:
        print(f"\n{tag} — {pool_caption}: PCA cluster 0 vs 1")
        wkl_cluster = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_rf_features(
            X_df=X_df,
            distribution_plot_results=dist_cluster,
            importance=importance,
            rank_col=rank_col,
            n_features=n_pool,
            n_bins_kl=N_BINS_KL,
        )
        csv_c = out_dir / f"wkl_pca_cluster_{key}.csv"
        wkl_cluster.to_csv(csv_c, index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_cluster,
            best_k=best_k,
            n_pool_features=n_pool,
            rank_col=rank_col,
            rank_xlabel=rank_xlabel,
            pool_caption=pool_caption,
            comparison_label="cluster 0 vs 1",
            split_caption="random train/test split",
            save_path=str(out_dir / f"wkl_pca_cluster_{key}.png"),
        )

        print(f"\n{tag} — {pool_caption}: KinCore inactive vs active")
        wkl_act = FeatureClassification.compute_active_vs_inactive_wasserstein_kl_for_rf_features(
            X_df=X_df,
            distribution_plot_results=dist_activation,
            importance=importance,
            rank_col=rank_col,
            n_features=n_pool,
            n_bins_kl=N_BINS_KL,
        )
        csv_a = out_dir / f"wkl_kincore_activation_{key}.csv"
        wkl_act.to_csv(csv_a, index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_act,
            best_k=best_k,
            n_pool_features=n_pool,
            rank_col=rank_col,
            rank_xlabel=rank_xlabel,
            pool_caption=pool_caption,
            comparison_label="inactive vs active",
            split_caption="random train/test split",
            save_path=str(out_dir / f"wkl_kincore_activation_{key}.png"),
        )
        results[key] = {"cluster": wkl_cluster, "activation": wkl_act}
        print(f"Saved CSVs: {csv_c.name}, {csv_a.name}")

    return results


wkl_sc = _run_wkl_plots(
    X_df=X_df_sc,
    dist_cluster=dist_sc_cluster,
    dist_activation=dist_sc_activation,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    out_dir=WKL_SC_DIR,
    tag="Side-chain",
)
print("\n✅ Side-chain Wasserstein / KL complete")

## 7.5 Cα matrix: train RF and reuse the same split seed

Load the Cα feature matrix (prefer filtered/corr-filtered if present), train a second RF with the same `train_size` / `random_state`, then repeat violins + W/KL.

In [ ]:
# Prefer filtered Cα if present; otherwise raw ca_feature_matrix.csv
_ca_candidates = [
    "ca_corr_filtered_feature_matrix.csv",
    "ca_filtered_feature_matrix.csv",
    "ca_feature_matrix.csv",
]
_ca_path = next((p for p in _ca_candidates if os.path.isfile(p)), None)
if _ca_path is None:
    raise FileNotFoundError(
        "No Cα feature matrix found. Expected one of: " + ", ".join(_ca_candidates)
    )
if _ca_path == "ca_feature_matrix.csv":
    print(f"⚠️  Using raw {_ca_path} (no ca_*filtered* matrix found)")

X_df_ca = pd.read_csv(_ca_path, index_col=0)

# Align to side-chain structures when possible
shared = [s for s in X_df_sc.index if s in X_df_ca.index]
if len(shared) < 10:
    # try normalized keys
    sc_norm = {FeatureClassification._normalize_structure_name(s): s for s in X_df_sc.index}
    ca_norm = {FeatureClassification._normalize_structure_name(s): s for s in X_df_ca.index}
    shared_keys = sorted(set(sc_norm) & set(ca_norm))
    shared = [ca_norm[k] for k in shared_keys]
    X_df_ca = X_df_ca.loc[shared]
    # remap SC-aligned labels via normalized names
    sc_to_label = {
        FeatureClassification._normalize_structure_name(s): y
        for s, y in zip(X_df_sc.index, y_sc)
    }
    y_ca = np.array([sc_to_label[FeatureClassification._normalize_structure_name(s)] for s in shared])
else:
    X_df_ca = X_df_ca.loc[shared]
    y_ca = np.asarray(y_sc[[list(X_df_sc.index).index(s) for s in shared]])

# Parse unique pairs from column names "i-j"
unique_pairs_ca = []
for col in X_df_ca.columns:
    parts = str(col).split("-")
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        unique_pairs_ca.append((int(parts[0]), int(parts[1])))
    else:
        unique_pairs_ca.append((col, col))

fully_conserved_ca = getattr(classifier, "fully_conserved", None) or []

classifier_ca = FeatureClassification(
    feature_matrix=X_df_ca.values.astype(float),
    labels=y_ca,
    unique_pairs=unique_pairs_ca,
    fully_conserved=fully_conserved_ca,
    structure_names=list(X_df_ca.index),
)
classifier_ca.split_data(train_size=0.9, random_state=42)
classifier_ca.train_model(n_estimators=100, random_state=42)
_ = classifier_ca.evaluate_model()
gini_mean_ca, _, _ = classifier_ca.compute_feature_importances()
perm_result_ca = classifier_ca.compute_permutation_importances(n_repeats=10, n_jobs=4)
perm_mean_ca = np.asarray(perm_result_ca.importances_mean, dtype=float)

Xk_ca = X_df_ca.values.astype(float)
feature_labels_ca = [str(c) for c in X_df_ca.columns]
split_labels_ca = np.asarray(classifier_ca.split_labels)
train_idx_ca = np.asarray(classifier_ca.train_idx)
test_idx_ca = np.asarray(classifier_ca.test_idx)
best_k_ca = Xk_ca.shape[1]

# KinCore labels for Cα structures
bio_labels_ca, _ = cluster_analyzer.load_kincore_labels(
    list(X_df_ca.index), kincore_file=KINCORE_CSV
)
bio_labels_ca = np.asarray(bio_labels_ca, dtype=float)
KINCORE_BIO_CSV_CA = str(WKL_CA_DIR / "kincore_bio_labels.csv")
pd.DataFrame({"structure": X_df_ca.index, "label": bio_labels_ca.astype(int)}).to_csv(
    KINCORE_BIO_CSV_CA, index=False
)

print(f"Cα matrix ({_ca_path}): {X_df_ca.shape[0]} × {X_df_ca.shape[1]}")
print(f"Train/val: {len(train_idx_ca)} / {len(test_idx_ca)}")
print(f"W/KL output: {WKL_CA_DIR}")

## 7.6 Cα: split violins (PCA cluster + KinCore)

In [ ]:
dist_ca_cluster = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
    X_df=X_df_ca,
    Xk=Xk_ca,
    feature_labels_all=feature_labels_ca,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    biological_labels_csv=KINCORE_BIO_CSV_CA,
    pca_cluster_labels_file=PCA_CLUSTER_LABELS,
    split_labels=split_labels_ca,
    train_idx=train_idx_ca,
    test_idx=test_idx_ca,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(Cα RF; k={best_k_ca})",
)

dist_ca_activation = FeatureClassification.plot_top_feature_distributions_by_activation(
    X_df=X_df_ca,
    Xk=Xk_ca,
    feature_labels_all=feature_labels_ca,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    biological_labels_csv=KINCORE_BIO_CSV_CA,
    split_labels=split_labels_ca,
    train_idx=train_idx_ca,
    test_idx=test_idx_ca,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(Cα RF; k={best_k_ca})",
)
print("✅ Cα distribution plots complete")

## 7.7 Cα: Wasserstein & KL (RF-ranked)

In [ ]:
wkl_ca = _run_wkl_plots(
    X_df=X_df_ca,
    dist_cluster=dist_ca_cluster,
    dist_activation=dist_ca_activation,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    out_dir=WKL_CA_DIR,
    tag="Cα",
)
print("\n✅ Cα Wasserstein / KL complete")
print("=" * 60)
print("✅ SECTION 7 COMPLETE (side-chain + Cα distributions and W/KL)")
print("=" * 60)

# 8. Side-chain vs Cα W/KL comparison  <a id="8"></a>

Compare separation strength of **side-chain** vs **Cα** distance features using the W/KL tables from §7:

1. **Overlay by RF rank** — plot Wasserstein and symmetric KL vs each modality’s own MDI/permutation rank (residue pairs need not match). Shows which distance type separates PCA clusters / KinCore classes more strongly among its top RF features.
2. **Shared residue pairs** — intersect canonical `i-j` pair IDs present in both SC and Cα W/KL tables; plot SC vs Cα metrics for the same pairs (ordered by mean RF rank).

Covers PCA cluster 0 vs 1 and KinCore inactive vs active, for both MDI and permutation rankings. Train and validation curves are shown. Outputs go to `Results/wkl_analysis/compare/`.


## 8.1 Overlay by rank + shared-pair comparison

Reuse `wkl_sc` / `wkl_ca` from §7 when present; otherwise reload CSVs from `Results/wkl_analysis/{sidechain,ca}/`.


In [ ]:
from pathlib import Path

WKL_COMPARE_DIR = WKL_ROOT / "compare"
WKL_COMPARE_DIR.mkdir(parents=True, exist_ok=True)

# Truncate overlay curves to this many top ranks (None => shorter of the two full lengths)
N_COMPARE_TOP = 100

RANK_JOBS = [
    ("mdi", "mdi_rank", "MDI (Gini) rank (1 = highest)"),
    ("perm", "perm_rank", "Permutation rank (1 = highest)"),
]
COMPARISONS = [
    ("cluster", "cluster", "cluster 0 vs 1"),
    ("activation", "activation", "inactive vs active"),
]


def _load_wkl_bundle(out_dir: Path):
    """Load {mdi,perm} × {cluster,activation} CSVs written by §7."""
    bundle = {}
    for key, _, _ in RANK_JOBS:
        cluster_csv = out_dir / f"wkl_pca_cluster_{key}.csv"
        act_csv = out_dir / f"wkl_kincore_activation_{key}.csv"
        if not cluster_csv.is_file() or not act_csv.is_file():
            raise FileNotFoundError(
                f"Missing W/KL CSVs under {out_dir} "
                f"(need {cluster_csv.name} and {act_csv.name}). Run §7 first."
            )
        bundle[key] = {
            "cluster": pd.read_csv(cluster_csv),
            "activation": pd.read_csv(act_csv),
        }
    return bundle


# Prefer in-memory §7 results
try:
    _ = wkl_sc
    _ = wkl_ca
except NameError:
    print("wkl_sc / wkl_ca not in memory — reloading from CSV…")
    wkl_sc = _load_wkl_bundle(WKL_SC_DIR)
    wkl_ca = _load_wkl_bundle(WKL_CA_DIR)

shared_counts = {}
for key, rank_col, rank_xlabel in RANK_JOBS:
    for comp_name, comp_key, comp_label in COMPARISONS:
        sc_df = wkl_sc[key][comp_key]
        ca_df = wkl_ca[key][comp_key]

        n_top = N_COMPARE_TOP
        if n_top is not None:
            n_top = min(int(n_top), len(sc_df), len(ca_df))
        else:
            n_top = min(len(sc_df), len(ca_df))

        print(f"\n── Overlay: {comp_label} | {key} (n_top={n_top}) ──")
        FeatureClassification.plot_sc_vs_ca_wkl_overlay_by_rank(
            sc_df,
            ca_df,
            rank_col=rank_col,
            comparison_label=comp_label,
            rank_xlabel=rank_xlabel,
            n_top=n_top,
            save_path=str(WKL_COMPARE_DIR / f"overlay_{comp_name}_{key}.png"),
        )

        aligned = FeatureClassification.align_shared_pair_wkl(
            sc_df, ca_df, rank_col=rank_col
        )
        csv_path = WKL_COMPARE_DIR / f"shared_pairs_{comp_name}_{key}.csv"
        aligned.to_csv(csv_path, index=False)
        n_shared = len(aligned)
        shared_counts[f"{comp_name}_{key}"] = n_shared
        print(f"Shared pairs ({comp_name}, {key}): {n_shared} → {csv_path.name}")

        FeatureClassification.plot_sc_vs_ca_wkl_shared_pairs(
            aligned,
            comparison_label=comp_label,
            save_path=str(WKL_COMPARE_DIR / f"shared_pairs_{comp_name}_{key}.png"),
        )

print("\n" + "=" * 60)
print("✅ SECTION 8 COMPLETE (SC vs Cα W/KL comparison)")
print(f"Outputs: {WKL_COMPARE_DIR}")
print("Shared-pair counts:", shared_counts)
print("=" * 60)
